# Sports Analytics Club: Intro EDA (Student Notebook)

Todays game plan:
- inspect the data
- filter to one season
- make a clean top-10 chart
- make an offense-vs-defense scatter
- optional challenge: build first-round matchup visuals from the 68-team field

## Local VS Code vs Google Colab Data Access

You can run this notebook in both places.

- **VS Code:** if your cloned repo is set up correctly, files should already be in `data/processed/club_share`.
- **Google Colab:** upload CSVs to `/content` (or a folder in `/content`) and point `COLAB_DATA_DIR` there.

Needed files (mens example):
- `m_team_aggregates_2005_2026.csv`
- `m_modeling_matchups_2026_all_possible.csv`

If you want womens data, switch `m_` to `w_` (womens team aggregates use the 2014-2026 span).

Tip: if paths fail in Colab, run `!ls /content` in a cell to see exactly where your files landed.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
# TODO: set to 'm' for mens or 'w' for womens
DATA_PREFIX = "___"

# TODO: set the season you want to analyze
SEASON = ___

# For Google Colab uploads, update this path if your files are in a subfolder
COLAB_DATA_DIR = Path("/content")

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    share_path = COLAB_DATA_DIR
else:
    project_root = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
    share_path = project_root / "data" / "processed" / "club_share"

# Team aggregate filename uses a different historical span for womens.
team_agg_year_span = "2005_2026" if DATA_PREFIX.lower() == "m" else "2014_2026"
team_agg_path = share_path / f"{DATA_PREFIX}_team_aggregates_{team_agg_year_span}.csv"
matchups_all_path = share_path / f"{DATA_PREFIX}_modeling_matchups_2026_all_possible.csv"

required_paths = [team_agg_path, matchups_all_path]
missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError("Missing files: " + ", ".join(missing_paths))

team_agg = pd.read_csv(team_agg_path)
matchups_all = pd.read_csv(matchups_all_path)

league_label = "Mens" if DATA_PREFIX.lower() == "m" else "Womens"
print(f"Loaded {league_label} data")
print(f"team_agg rows: {len(team_agg):,}")
print(f"modeling matchups rows: {len(matchups_all):,}")

## 1) Data Inspection
Fast sanity check first.

If this part looks wrong, every chart after this will be wrong too.

In [ ]:
# TODO: preview the first 5 rows
# Hint: use the .head() method
display(team_agg.___)

# TODO: print the shape
# Hint: f-string + team_agg.shape
print(f"Shape: {___}")

# TODO: show column types and non-null counts
# Hint: DataFrame.info()
team_agg.___()

## 2) Filter to One Season + Missing Values

In [ ]:
# TODO: filter team_agg to only SEASON
# Hint: boolean filtering with team_agg['Season'] == SEASON defined above
season_df = ___

print(f"Rows in season {SEASON}: {len(season_df):,}")

key_cols = ["TeamName", "Final_Massey", "avg_off_rating", "avg_def_rating", "SeedNum"]

# TODO: count missing values in key_cols
# Hint: isna().sum(), then sort_values(ascending=False) for most-missing first
missing_summary = ___
display(missing_summary.to_frame(name="missing_count"))

## 3) Bar Chart: Top 10 Final Massey Teams in 2026
Rank teams by Final Massey for the season you selected.

In [ ]:
# TODO: keep rows with non-missing TeamName and Final_Massey
# TODO: sort descending by Final_Massey
# TODO: keep top 10
bar_df = ___

plt.figure(figsize=(10, 6))
# TODO: build a seaborn horizontal barplot
sns.barplot(data=___, x=___, y=___, palette="Blues_r")
plt.title(f"Top 10 Teams by Final Massey ({SEASON})")
plt.xlabel("Final Massey")
plt.ylabel("Team")
plt.tight_layout()
plt.show()

## 4) Scatter Plot Progression: Offense vs Defense
This section is a step-by-step upgrade path, not a one-shot final chart.

Steps:
1. Plot all teams.
2. Flip defense so lower defensive rating becomes higher on the y-axis.
3. Keep only tournament teams.
4. Label top 10 teams using combined score (`avg_off_rating - avg_def_rating`).

In [ ]:
scatter_base = season_df.dropna(
    subset=["avg_off_rating", "avg_def_rating", "Final_Massey", "TeamName"]
).copy()

# Step 1: plot all teams as-is.
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=scatter_base,
    x="avg_off_rating",
    y="avg_def_rating",
    alpha=0.6,
    s=55,
    color="#4C78A8",
)
plt.title(f"Step 1: All Teams (Raw Offense vs Defense) - {SEASON}")
plt.xlabel("Average Offensive Rating")
plt.ylabel("Average Defensive Rating")
plt.tight_layout()
plt.show()

In [ ]:
# Step 2: flip defense so lower defensive rating appears higher.
# TODO: create defense_better_high = -avg_def_rating
scatter_base["defense_better_high"] = ___

plt.figure(figsize=(10, 7))
# TODO: use y='defense_better_high'
sns.scatterplot(data=scatter_base, x="avg_off_rating", y=___, alpha=0.6, s=55, color="#59A14F")
plt.title(f"Step 2: Flipped Defense Axis (Higher is Better) - {SEASON}")
plt.xlabel("Average Offensive Rating")
plt.ylabel("- Average Defensive Rating")
plt.tight_layout()
plt.show()

In [ ]:
# Step 3: filter to tournament teams.
# Hint: SeedNum is present for tournament teams.
seednum = pd.to_numeric(scatter_base.get("SeedNum"), errors="coerce")
tourney_df = scatter_base[___].copy()

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=tourney_df,
    x="avg_off_rating",
    y="defense_better_high",
    alpha=0.7,
    s=65,
    color="#F28E2B",
)
plt.title(f"Step 3: Tournament Teams Only - {SEASON}")
plt.xlabel("Average Offensive Rating")
plt.ylabel("- Average Defensive Rating")
plt.tight_layout()
plt.show()

In [ ]:
# Step 4: label top 10 by combined score = off - def.
# TODO: create combined_score and choose top 10
# Hint: use pd.to_numeric(..., errors='coerce') before subtracting
tourney_df["combined_score"] = ___
top10_combined = ___

plt.figure(figsize=(11, 8))
ax = sns.scatterplot(
    data=tourney_df,
    x="avg_off_rating",
    y="defense_better_high",
    hue="combined_score",
    palette="viridis",
    alpha=0.75,
    s=70,
)

# TODO: loop over top10_combined and label each team with TeamName
for _, row in top10_combined.iterrows():
    ax.text(
        row["avg_off_rating"] + 0.03,
        row["defense_better_high"] + 0.03,
        row["TeamName"],
        fontsize=8,
        fontweight="bold",
    )

plt.title(f"Step 4: Tournament Teams + Top 10 Combined Score Labels - {SEASON}")
plt.xlabel("Average Offensive Rating")
plt.ylabel("- Average Defensive Rating")
plt.tight_layout()
plt.show()

## Next Steps
- Create any additional visuals you want using the columns in the team aggregate data. You can refer to the advanced notebook for examples and how to get logos and colors. 
- Move onto matchup visuals in the next section

## 5) Optional Challenge: First-Round Matchup Visuals (Precomputed)

Make visuals from the matchup output.

Example idea: plot games where `Diff_Final_Massey < 0` (lower seed has better Final Massey).

In [ ]:
round1_path = share_path / f"{DATA_PREFIX}_first_round_matchups_2026_68team_candidates.csv"
if not round1_path.exists():
    raise FileNotFoundError(
        f"Could not find precomputed first-round file: {round1_path}\n"
        "Run the first-round export preprocessing first so this club_share file exists."
    )

round1_with_modeling = pd.read_csv(round1_path)

required_cols = [
    "Region",
    "FavoriteSeed",
    "UnderdogSeed",
    "FavoriteSeedCode",
    "UnderdogSeedCode",
    "FavoriteTeam",
    "UnderdogTeam",
    "Diff_Final_Massey",
    "HasPlayInPath",
    "InModelingFile",
    "MatchupLabel",
]
missing_cols = [c for c in required_cols if c not in round1_with_modeling.columns]
if missing_cols:
    raise RuntimeError("Precomputed file is missing columns: " + ", ".join(missing_cols))

print(f"Loaded precomputed first-round file: {round1_path.name}")
print(f"Rows: {len(round1_with_modeling):,}")
print(f"Rows with play-in paths: {int(round1_with_modeling['HasPlayInPath'].sum()):,}")
print(f"Rows linked to modeling file: {int(round1_with_modeling['InModelingFile'].sum()):,}")

display_cols = [
    "Region",
    "FavoriteSeedCode",
    "FavoriteTeam",
    "UnderdogSeedCode",
    "UnderdogTeam",
    "Diff_Final_Massey",
    "HasPlayInPath",
    "InModelingFile",
]
display(
    round1_with_modeling.sort_values(
        ["Region", "FavoriteSeed", "UnderdogSeed", "FavoriteSeedCode", "UnderdogSeedCode"]
    )[display_cols].head(40)
)

In [ ]:
# TODO challenge: build your own first-round visual from round1_with_modeling.
# Ideas:
# 1) only HasPlayInPath == True
# 2) top absolute Diff_Final_Massey games
# 3) one region at a time

# Example starter visual (already complete): lower seed has better Final Massey.
example_df = (
    round1_with_modeling.dropna(subset=["Diff_Final_Massey"])
    .query("Diff_Final_Massey < 0")
    .sort_values("Diff_Final_Massey")
    .head(12)
    .copy()
)

if example_df.empty:
    print("No negative Diff_Final_Massey matchups found for this setup.")
else:
    plt.figure(figsize=(12, 7))
    sns.barplot(data=example_df, x="Diff_Final_Massey", y="MatchupLabel", palette="rocket")
    plt.title(f"Example: Lower Seed with Better Final Massey (2026)")
    plt.xlabel("Diff_Final_Massey (Favorite - Underdog)")
    plt.ylabel("Matchup")
    plt.tight_layout()
    plt.show()